# Reassessment Notebook: Time Holdout + Stepwise Regression

This notebook performs:
1. Historical data loading with a **time-based holdout split** (latest pricing date as test set).
2. **Stepwise regression** where `owners_estimate` is the dependent variable.
3. Extraction of a **best feature shortlist** to feed downstream machine learning models.

In [19]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm.auto import tqdm

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
cell_pbar = tqdm(total=1, desc='Cell 1/9: Load data', leave=False)

# Config
DATA_PATH = Path('/Users/arifpras/Library/CloudStorage/Dropbox/perisai/dashboard/model/dbtrain.csv')
TARGET_COL = 'owners_estimate'
DATE_COL = 'tanggal_lelang_pricing'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Data file not found: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
print(f'Loaded data: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

required_cols = [TARGET_COL, DATE_COL]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')

cell_pbar.update(1)
cell_pbar.close()

df.head()

Cell 1/9: Load data:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded data: 1,056 rows x 28 columns


,tanggal_lelang_pricing,code,tenor,ust_10y,ust_2y,vix,move,dxy,ndf_1m,ndf_12m,...,tenor_x_ust10y,vix_x_move,prev_owners_estimate_code,prev_way_awarded_code,days_since_prev_auction_code,auctions_since_prev_code,prev_bid_to_cover_ratio_code,prev_total_penawaran_code,prev_total_penawaran_diterima_code,owners_estimate
0,07/01/2020,bm05,5.431896,1.81,1.546,13.85,62.79,96.671,0.150602,3.95152,...,9.831732,869.6415,NaN,NaN,NaN,0,NaN,NaN,NaN,6.39
1,07/01/2020,bm20,20.265572,1.81,1.546,13.85,62.79,96.671,0.150602,3.95152,...,36.680684,869.6415,NaN,NaN,NaN,0,NaN,NaN,NaN,7.54
2,07/01/2020,bm10,10.683094,1.81,1.546,13.85,62.79,96.671,0.150602,3.95152,...,19.336400,869.6415,NaN,NaN,NaN,0,NaN,NaN,NaN,7.10
3,07/01/2020,bm30,28.347707,1.81,1.546,13.85,62.79,96.671,0.150602,3.95152,...,51.309350,869.6415,NaN,NaN,NaN,0,NaN,NaN,NaN,7.72
4,07/01/2020,spn03m,0.251882,1.81,1.546,13.85,62.79,96.671,0.150602,3.95152,...,0.455907,869.6415,NaN,NaN,NaN,0,NaN,NaN,NaN,4.65


In [3]:
cell_pbar = tqdm(total=1, desc='Cell 2/9: Time split', leave=False)

# Time-based holdout split: latest pricing date as test
work = df.copy()
work[DATE_COL + '_dt'] = pd.to_datetime(work[DATE_COL], dayfirst=True, errors='coerce')
work[TARGET_COL] = pd.to_numeric(work[TARGET_COL], errors='coerce')

work = work.dropna(subset=[DATE_COL + '_dt', TARGET_COL]).copy()
if work.empty:
    raise ValueError('No usable rows after date/target cleaning.')

latest_date = work[DATE_COL + '_dt'].max()
train_df = work[work[DATE_COL + '_dt'] < latest_date].copy()
test_df = work[work[DATE_COL + '_dt'] == latest_date].copy()

if train_df.empty or test_df.empty:
    raise ValueError('Train or test split is empty. Check date distribution.')

print(f'Latest pricing date (test): {latest_date.date()}')
print(f'Train rows: {len(train_df):,}')
print(f'Test rows : {len(test_df):,}')

cell_pbar.update(1)
cell_pbar.close()

Latest pricing date (test): 2026-02-03
Train rows: 1,047
Test rows : 9


In [4]:
cell_pbar = tqdm(total=4, desc='Cell 3/9: Build matrices', leave=False)

# Build design matrices for regression
drop_non_features = [TARGET_COL, DATE_COL, DATE_COL + '_dt']

X_train_raw = train_df.drop(columns=[c for c in drop_non_features if c in train_df.columns], errors='ignore').copy()
X_test_raw = test_df.drop(columns=[c for c in drop_non_features if c in test_df.columns], errors='ignore').copy()
y_train = train_df[TARGET_COL].astype(float).copy()
y_test = test_df[TARGET_COL].astype(float).copy()
cell_pbar.update(1)

# One-hot encode categoricals so stepwise OLS can work on numeric columns
X_train_enc = pd.get_dummies(X_train_raw, drop_first=True, prefix_sep='__')
X_test_enc = pd.get_dummies(X_test_raw, drop_first=True, prefix_sep='__')

# Align test columns to train columns
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

# Numeric coercion + cleanup
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_test_enc = X_test_enc.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
cell_pbar.update(1)

# Drop constant/all-null columns on train
valid_cols = []
for col in tqdm(X_train_enc.columns, desc='Checking candidate columns', leave=False):
    s = X_train_enc[col]
    if s.notna().sum() == 0:
        continue
    if s.nunique(dropna=True) <= 1:
        continue
    valid_cols.append(col)

X_train_enc = X_train_enc[valid_cols].fillna(X_train_enc[valid_cols].median(numeric_only=True))
X_test_enc = X_test_enc[valid_cols].fillna(X_train_enc.median(numeric_only=True))
cell_pbar.update(1)

# Force float matrix for statsmodels
X_train_enc = X_train_enc.astype(float)
X_test_enc = X_test_enc.astype(float)

print(f'Encoded candidate features: {X_train_enc.shape[1]:,}')
cell_pbar.update(1)
cell_pbar.close()

Encoded candidate features: 34


In [5]:
cell_pbar = tqdm(total=1, desc='Cell 4/9: Define stepwise', leave=False)

def stepwise_selection(X, y, threshold_in=0.01, threshold_out=0.05, max_iter=1000, verbose=True):
    """
    Forward-backward stepwise feature selection based on OLS p-values.
    Returns selected feature names in inclusion order.
    """
    included = []

    for _ in tqdm(range(max_iter), desc='Stepwise iterations', leave=False):
        changed = False

        # Forward step
        excluded = [col for col in X.columns if col not in included]
        new_pvals = pd.Series(dtype=float)

        for col in excluded:
            X_model = sm.add_constant(X[included + [col]], has_constant='add')
            model = sm.OLS(y, X_model).fit()
            new_pvals.loc[col] = model.pvalues.get(col, np.nan)

        if not new_pvals.empty:
            best_pval = new_pvals.min()
            best_feature = new_pvals.idxmin()
            if pd.notna(best_pval) and best_pval < threshold_in:
                included.append(best_feature)
                changed = True
                if verbose:
                    print(f'Add  : {best_feature:<40} p={best_pval:.6g}')

        # Backward step
        if included:
            X_model = sm.add_constant(X[included], has_constant='add')
            model = sm.OLS(y, X_model).fit()

            pvalues = model.pvalues.drop('const', errors='ignore')
            if not pvalues.empty:
                worst_pval = pvalues.max()
                worst_feature = pvalues.idxmax()

                if pd.notna(worst_pval) and worst_pval > threshold_out:
                    included.remove(worst_feature)
                    changed = True
                    if verbose:
                        print(f'Drop : {worst_feature:<40} p={worst_pval:.6g}')

        if not changed:
            if verbose:
                print('No more changes. Stepwise selection converged.')
            break

    return included

cell_pbar.update(1)
cell_pbar.close()

In [6]:
cell_pbar = tqdm(total=1, desc='Cell 5/9: Run stepwise', leave=False)

selected_features = stepwise_selection(
    X_train_enc,
    y_train,
    threshold_in=0.01,
    threshold_out=0.05,
    verbose=True
)

print('\nSelected encoded features:', len(selected_features))
cell_pbar.update(1)
cell_pbar.close()
selected_features[:20]

Cell 5/9: Run stepwise:   0%|          | 0/1 [00:00<?, ?it/s]

Add  : prev_owners_estimate_code                p=0
Add  : vix_x_move                               p=6.5558e-12
Add  : tenor                                    p=0.000105533
Add  : dxy                                      p=0.000733418
Add  : code__spn01m                             p=0.00132622


Add  : tenor_x_ust10y                           p=0.00151516
Add  : ust_10y                                  p=3.42508e-05
Drop : dxy                                      p=0.0527742
Add  : ndf_12m                                  p=3.36882e-06
Add  : code__spn03m                             p=3.15067e-05


Add  : code__spn12m                             p=6.52467e-10
Add  : prev_total_penawaran_code                p=1.28143e-07


Add  : indo_10y                                 p=0.000726273
Add  : code__bm10                               p=0.00417542
No more changes. Stepwise selection converged.

Selected encoded features: 12


['prev_owners_estimate_code',
 'vix_x_move',
 'tenor',
 'code__spn01m',
 'tenor_x_ust10y',
 'ust_10y',
 'ndf_12m',
 'code__spn03m',
 'code__spn12m',
 'prev_total_penawaran_code',
 'indo_10y',
 'code__bm10']

In [7]:
cell_pbar = tqdm(total=2, desc='Cell 6/9: Fit and evaluate', leave=False)

if not selected_features:
    raise ValueError('No features selected by stepwise regression. Consider relaxing thresholds.')

X_train_sel = sm.add_constant(X_train_enc[selected_features], has_constant='add')
X_test_sel = sm.add_constant(X_test_enc[selected_features], has_constant='add')

ols_model = sm.OLS(y_train, X_train_sel).fit()
y_pred_test = ols_model.predict(X_test_sel)
cell_pbar.update(1)

mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mape = (np.abs((y_test - y_pred_test) / np.clip(np.abs(y_test), 1e-8, None))).mean() * 100
r2 = r2_score(y_test, y_pred_test)

metrics_df = pd.DataFrame([{
    'test_date': latest_date.date(),
    'n_train': len(train_df),
    'n_test': len(test_df),
    'selected_features': len(selected_features),
    'MAE': mae,
    'RMSE': rmse,
    'MAPE_pct': mape,
    'R2': r2
}])

cell_pbar.update(1)
cell_pbar.close()
metrics_df

,test_date,n_train,n_test,selected_features,MAE,RMSE,MAPE_pct,R2
0,2026-02-03,1047,9,12,0.125255,0.147928,2.359814,0.977845


In [8]:
cell_pbar = tqdm(total=1, desc='Cell 7/9: Map base features', leave=False)

# Map encoded variables back to base feature names for ML shortlist
base_features = sorted({feat.split('__')[0] for feat in selected_features})

feature_shortlist_df = pd.DataFrame({
    'base_feature_for_ml': base_features
})

print(f'Base features recommended for ML: {len(base_features)}')
cell_pbar.update(1)
cell_pbar.close()
feature_shortlist_df.head(50)

Base features recommended for ML: 9


,base_feature_for_ml
0,code
1,indo_10y
2,ndf_12m
3,prev_owners_estimate_code
4,prev_total_penawaran_code
5,tenor
6,tenor_x_ust10y
7,ust_10y
8,vix_x_move


In [20]:
cell_pbar = tqdm(total=4, desc='Cell 8/11: LightGBM holdout', leave=False)

# LightGBM reassessment using selected base features
lgbm_df = pd.read_csv(DATA_PATH)
cell_pbar.update(1)

lgbm_work = lgbm_df.copy()
lgbm_work[DATE_COL + '_dt'] = pd.to_datetime(lgbm_work[DATE_COL], dayfirst=True, errors='coerce')
lgbm_work[TARGET_COL] = pd.to_numeric(lgbm_work[TARGET_COL], errors='coerce')
lgbm_work = lgbm_work.dropna(subset=[DATE_COL + '_dt', TARGET_COL]).copy()
if lgbm_work.empty:
    raise ValueError('No usable rows after date/target cleaning for LightGBM block.')

lgbm_latest_date = lgbm_work[DATE_COL + '_dt'].max()
lgbm_train_df = lgbm_work[lgbm_work[DATE_COL + '_dt'] < lgbm_latest_date].copy()
lgbm_test_df = lgbm_work[lgbm_work[DATE_COL + '_dt'] == lgbm_latest_date].copy()
if lgbm_train_df.empty or lgbm_test_df.empty:
    raise ValueError('Train or test split is empty for LightGBM block. Check date distribution.')
cell_pbar.update(1)

lgbm_feature_cols = [col for col in base_features if col in lgbm_train_df.columns]
if not lgbm_feature_cols:
    raise ValueError('No shortlisted base features found in dbtrain.csv for LightGBM.')

X_train_lgb_raw = lgbm_train_df[lgbm_feature_cols].copy()
X_test_lgb_raw = lgbm_test_df[lgbm_feature_cols].copy()
y_train_lgb = lgbm_train_df[TARGET_COL].astype(float).copy()
y_test_lgb = lgbm_test_df[TARGET_COL].astype(float).copy()

X_train_lgb = pd.get_dummies(X_train_lgb_raw, drop_first=True, prefix_sep='__')
X_test_lgb = pd.get_dummies(X_test_lgb_raw, drop_first=True, prefix_sep='__')
X_train_lgb, X_test_lgb = X_train_lgb.align(X_test_lgb, join='left', axis=1, fill_value=0)

X_train_lgb = X_train_lgb.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_test_lgb = X_test_lgb.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_train_lgb = X_train_lgb.fillna(X_train_lgb.median(numeric_only=True))
X_test_lgb = X_test_lgb.fillna(X_train_lgb.median(numeric_only=True))
cell_pbar.update(1)

lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm_model.fit(X_train_lgb, y_train_lgb)
y_pred_lgb_test = np.asarray(lgbm_model.predict(X_test_lgb), dtype=float).ravel()

lgbm_mae = mean_absolute_error(y_test_lgb, y_pred_lgb_test)
lgbm_rmse = np.sqrt(mean_squared_error(y_test_lgb, y_pred_lgb_test))
lgbm_mape = (np.abs((y_test_lgb - y_pred_lgb_test) / np.clip(np.abs(y_test_lgb), 1e-8, None))).mean() * 100
lgbm_r2 = r2_score(y_test_lgb, y_pred_lgb_test)

lgbm_metrics_df = pd.DataFrame([{
    'test_date': lgbm_latest_date.date(),
    'n_train': len(lgbm_train_df),
    'n_test': len(lgbm_test_df),
    'n_base_features': len(lgbm_feature_cols),
    'MAE': lgbm_mae,
    'RMSE': lgbm_rmse,
    'MAPE_pct': lgbm_mape,
    'R2': lgbm_r2
}])

print(f'LightGBM holdout date (test): {lgbm_latest_date.date()}')
print(f'LightGBM features used: {len(lgbm_feature_cols)}')
cell_pbar.update(1)
cell_pbar.close()
lgbm_metrics_df

Cell 8/11: LightGBM holdout:   0%|          | 0/4 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1592
[LightGBM] [Info] Number of data points in the train set: 1047, number of used features: 15
[LightGBM] [Info] Start training from score 6.169024


LightGBM holdout date (test): 2026-02-03
LightGBM features used: 9


,test_date,n_train,n_test,n_base_features,MAE,RMSE,MAPE_pct,R2
0,2026-02-03,1047,9,9,0.101464,0.123573,1.883643,0.98454


In [21]:
cell_pbar = tqdm(total=3, desc='Cell 9/11: XGBoost holdout', leave=False)

# XGBoost reassessment using the same holdout split and encoded feature matrix
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)
xgb_model.fit(X_train_lgb, y_train_lgb)
y_pred_xgb_test = np.asarray(xgb_model.predict(X_test_lgb), dtype=float).ravel()
cell_pbar.update(1)

xgb_mae = mean_absolute_error(y_test_lgb, y_pred_xgb_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test_lgb, y_pred_xgb_test))
xgb_mape = (np.abs((y_test_lgb - y_pred_xgb_test) / np.clip(np.abs(y_test_lgb), 1e-8, None))).mean() * 100
xgb_r2 = r2_score(y_test_lgb, y_pred_xgb_test)
cell_pbar.update(1)

xgb_metrics_df = pd.DataFrame([{
    'test_date': lgbm_latest_date.date(),
    'n_train': len(lgbm_train_df),
    'n_test': len(lgbm_test_df),
    'n_base_features': len(lgbm_feature_cols),
    'MAE': xgb_mae,
    'RMSE': xgb_rmse,
    'MAPE_pct': xgb_mape,
    'R2': xgb_r2
}])

print(f'XGBoost holdout date (test): {lgbm_latest_date.date()}')
print(f'XGBoost features used: {len(lgbm_feature_cols)} base / {X_train_lgb.shape[1]} encoded')
cell_pbar.update(1)
cell_pbar.close()
xgb_metrics_df

XGBoost holdout date (test): 2026-02-03
XGBoost features used: 9 base / 17 encoded


,test_date,n_train,n_test,n_base_features,MAE,RMSE,MAPE_pct,R2
0,2026-02-03,1047,9,9,0.10066,0.13716,2.005925,0.980953


In [22]:
cell_pbar = tqdm(total=3, desc='Cell 10/14: CatBoost holdout', leave=False)

# CatBoost reassessment using the same holdout split and encoded feature matrix
cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_seed=42,
    verbose=False
)
cat_model.fit(X_train_lgb, y_train_lgb)
y_pred_cat_test = np.asarray(cat_model.predict(X_test_lgb), dtype=float).ravel()
cell_pbar.update(1)

cat_mae = mean_absolute_error(y_test_lgb, y_pred_cat_test)
cat_rmse = np.sqrt(mean_squared_error(y_test_lgb, y_pred_cat_test))
cat_mape = (np.abs((y_test_lgb - y_pred_cat_test) / np.clip(np.abs(y_test_lgb), 1e-8, None))).mean() * 100
cat_r2 = r2_score(y_test_lgb, y_pred_cat_test)
cell_pbar.update(1)

cat_metrics_df = pd.DataFrame([{
    'test_date': lgbm_latest_date.date(),
    'n_train': len(lgbm_train_df),
    'n_test': len(lgbm_test_df),
    'n_base_features': len(lgbm_feature_cols),
    'MAE': cat_mae,
    'RMSE': cat_rmse,
    'MAPE_pct': cat_mape,
    'R2': cat_r2
}])

print(f'CatBoost holdout date (test): {lgbm_latest_date.date()}')
print(f'CatBoost features used: {len(lgbm_feature_cols)} base / {X_train_lgb.shape[1]} encoded')
cell_pbar.update(1)
cell_pbar.close()
cat_metrics_df

CatBoost holdout date (test): 2026-02-03
CatBoost features used: 9 base / 17 encoded


,test_date,n_train,n_test,n_base_features,MAE,RMSE,MAPE_pct,R2
0,2026-02-03,1047,9,9,0.11448,0.144136,2.220312,0.978966


In [23]:
cell_pbar = tqdm(total=3, desc='Cell 11/14: RandomForest holdout', leave=False)

# Random Forest reassessment using the same holdout split and encoded feature matrix
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_lgb, y_train_lgb)
y_pred_rf_test = np.asarray(rf_model.predict(X_test_lgb), dtype=float).ravel()
cell_pbar.update(1)

rf_mae = mean_absolute_error(y_test_lgb, y_pred_rf_test)
rf_rmse = np.sqrt(mean_squared_error(y_test_lgb, y_pred_rf_test))
rf_mape = (np.abs((y_test_lgb - y_pred_rf_test) / np.clip(np.abs(y_test_lgb), 1e-8, None))).mean() * 100
rf_r2 = r2_score(y_test_lgb, y_pred_rf_test)
cell_pbar.update(1)

rf_metrics_df = pd.DataFrame([{
    'test_date': lgbm_latest_date.date(),
    'n_train': len(lgbm_train_df),
    'n_test': len(lgbm_test_df),
    'n_base_features': len(lgbm_feature_cols),
    'MAE': rf_mae,
    'RMSE': rf_rmse,
    'MAPE_pct': rf_mape,
    'R2': rf_r2
}])

print(f'RandomForest holdout date (test): {lgbm_latest_date.date()}')
print(f'RandomForest features used: {len(lgbm_feature_cols)} base / {X_train_lgb.shape[1]} encoded')
cell_pbar.update(1)
cell_pbar.close()
rf_metrics_df

RandomForest holdout date (test): 2026-02-03
RandomForest features used: 9 base / 17 encoded


,test_date,n_train,n_test,n_base_features,MAE,RMSE,MAPE_pct,R2
0,2026-02-03,1047,9,9,0.12684,0.16807,2.580511,0.971401


In [24]:
cell_pbar = tqdm(total=3, desc='Cell 12/14: Quantile regression holdout', leave=False)

# Quantile regression (tau=0.5) reassessment using stepwise-selected encoded features
qr_tau = 0.5
qr_model = sm.QuantReg(y_train, X_train_sel).fit(q=qr_tau, max_iter=5000)
y_pred_qr_test = np.asarray(qr_model.predict(X_test_sel), dtype=float).ravel()
cell_pbar.update(1)

qr_mae = mean_absolute_error(y_test, y_pred_qr_test)
qr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_qr_test))
qr_mape = (np.abs((y_test - y_pred_qr_test) / np.clip(np.abs(y_test), 1e-8, None))).mean() * 100
qr_r2 = r2_score(y_test, y_pred_qr_test)
cell_pbar.update(1)

qr_metrics_df = pd.DataFrame([{
    'test_date': latest_date.date(),
    'n_train': len(train_df),
    'n_test': len(test_df),
    'n_base_features': len(base_features),
    'quantile_tau': qr_tau,
    'MAE': qr_mae,
    'RMSE': qr_rmse,
    'MAPE_pct': qr_mape,
    'R2': qr_r2
}])

print(f'Quantile Regression holdout date (test): {latest_date.date()}')
print(f'Quantile tau: {qr_tau}')
cell_pbar.update(1)
cell_pbar.close()
qr_metrics_df

Quantile Regression holdout date (test): 2026-02-03
Quantile tau: 0.5


,test_date,n_train,n_test,n_base_features,quantile_tau,MAE,RMSE,MAPE_pct,R2
0,2026-02-03,1047,9,9,0.5,0.424942,0.645208,9.240014,0.578531


In [25]:
cell_pbar = tqdm(total=2, desc='Cell 13/14: Leaderboard', leave=False)

metrics_ols = metrics_df.rename(columns={'selected_features': 'n_base_features'}).copy()
metrics_ols['model'] = 'OLS Stepwise'

metrics_lgbm = lgbm_metrics_df.copy()
metrics_lgbm['model'] = 'LightGBM'

metrics_xgb = xgb_metrics_df.copy()
metrics_xgb['model'] = 'XGBoost'

metrics_cat = cat_metrics_df.copy()
metrics_cat['model'] = 'CatBoost'

metrics_rf = rf_metrics_df.copy()
metrics_rf['model'] = 'RandomForest'

metrics_qr = qr_metrics_df.copy()
metrics_qr['model'] = 'QuantileReg_tau0.5'

leaderboard_df = pd.concat(
    [metrics_ols, metrics_lgbm, metrics_xgb, metrics_cat, metrics_rf, metrics_qr],
    ignore_index=True,
    sort=False
)

leaderboard_df = leaderboard_df[[
    'model', 'test_date', 'n_train', 'n_test', 'n_base_features',
    'MAE', 'RMSE', 'MAPE_pct', 'R2'
]].sort_values(by=['RMSE', 'MAE'], ascending=[True, True]).reset_index(drop=True)

cell_pbar.update(1)
print('Model leaderboard (lower RMSE/MAE is better):')
cell_pbar.update(1)
cell_pbar.close()
leaderboard_df

Model leaderboard (lower RMSE/MAE is better):


,model,test_date,n_train,n_test,n_base_features,MAE,RMSE,MAPE_pct,R2
0,LightGBM,2026-02-03,1047,9,9,0.101464,0.123573,1.883643,0.984540
1,XGBoost,2026-02-03,1047,9,9,0.100660,0.137160,2.005925,0.980953
2,CatBoost,2026-02-03,1047,9,9,0.114480,0.144136,2.220312,0.978966
3,OLS Stepwise,2026-02-03,1047,9,12,0.125255,0.147928,2.359814,0.977845
4,RandomForest,2026-02-03,1047,9,9,0.126840,0.168070,2.580511,0.971401
5,QuantileReg_tau0.5,2026-02-03,1047,9,9,0.424942,0.645208,9.240014,0.578531


In [27]:
cell_pbar = tqdm(total=4, desc='Cell 14/15: Predict dbpredict', leave=False)

PREDICT_PATH = DATA_PATH.parent / 'dbpredict.csv'
if not PREDICT_PATH.exists():
    raise FileNotFoundError(f'Prediction file not found: {PREDICT_PATH.resolve()}')

pred_df = pd.read_csv(PREDICT_PATH)
pred_work = pred_df.copy()
cell_pbar.update(1)

if DATE_COL in pred_work.columns:
    pred_work[DATE_COL + '_dt'] = pd.to_datetime(pred_work[DATE_COL], dayfirst=True, errors='coerce')

# OLS/Quantile prediction matrix (stepwise-selected encoded features)
drop_non_features_pred = [TARGET_COL, DATE_COL, DATE_COL + '_dt']
X_pred_raw = pred_work.drop(columns=[c for c in drop_non_features_pred if c in pred_work.columns], errors='ignore').copy()
X_pred_enc = pd.get_dummies(X_pred_raw, drop_first=True, prefix_sep='__')
X_pred_enc = X_pred_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_pred_enc = X_pred_enc.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_pred_enc = X_pred_enc.fillna(X_train_enc.median(numeric_only=True)).astype(float)

X_pred_sel = sm.add_constant(X_pred_enc[selected_features], has_constant='add')

pred_ols = np.asarray(ols_model.predict(X_pred_sel), dtype=float).ravel()
pred_qr = np.asarray(qr_model.predict(X_pred_sel), dtype=float).ravel()
cell_pbar.update(1)

# Tree-model prediction matrix (base feature shortlist -> encoded alignment)
X_pred_lgb_raw = pred_work.reindex(columns=lgbm_feature_cols)
X_pred_lgb = pd.get_dummies(X_pred_lgb_raw, drop_first=True, prefix_sep='__')
X_pred_lgb = X_pred_lgb.reindex(columns=X_train_lgb.columns, fill_value=0)
X_pred_lgb = X_pred_lgb.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_pred_lgb = X_pred_lgb.fillna(X_train_lgb.median(numeric_only=True)).astype(float)

pred_lgbm = np.asarray(lgbm_model.predict(X_pred_lgb), dtype=float).ravel()
pred_xgb = np.asarray(xgb_model.predict(X_pred_lgb), dtype=float).ravel()
pred_cat = np.asarray(cat_model.predict(X_pred_lgb), dtype=float).ravel()
pred_rf = np.asarray(rf_model.predict(X_pred_lgb), dtype=float).ravel()
cell_pbar.update(1)

id_cols = [col for col in [DATE_COL, 'seri', 'tenor', 'instrument', 'id'] if col in pred_df.columns]
prediction_comparison_df = pred_df[id_cols].copy() if id_cols else pd.DataFrame(index=pred_df.index)

prediction_comparison_df['pred_ols_stepwise'] = pred_ols
prediction_comparison_df['pred_lightgbm'] = pred_lgbm
prediction_comparison_df['pred_xgboost'] = pred_xgb
prediction_comparison_df['pred_catboost'] = pred_cat
prediction_comparison_df['pred_random_forest'] = pred_rf
prediction_comparison_df['pred_quantile_tau_0_5'] = pred_qr

print(f'Prediction rows scored: {len(prediction_comparison_df):,}')
cell_pbar.update(1)
cell_pbar.close()
prediction_comparison_df

Prediction rows scored: 9


,tanggal_lelang_pricing,tenor,pred_ols_stepwise,pred_lightgbm,pred_xgboost,pred_catboost,pred_random_forest,pred_quantile_tau_0_5
0,18/02/2026,0.087671,4.220448,4.450266,4.532050,4.554587,4.519180,5.332037
1,18/02/2026,0.252055,4.527677,4.662560,4.679827,4.514280,4.588440,5.355166
2,18/02/2026,0.961644,4.706004,4.797099,4.785763,4.731214,4.734860,5.462498
3,18/02/2026,5.068918,5.770871,5.769527,5.683437,5.728731,5.679080,6.071012
4,18/02/2026,10.154057,6.376431,6.332526,6.220516,6.285317,6.312340,6.374520
5,18/02/2026,14.488045,6.560334,6.556121,6.527014,6.569884,6.521560,6.431430
6,18/02/2026,19.488022,6.609975,6.603430,6.622194,6.633834,6.576324,6.536635
7,18/02/2026,28.403134,6.733233,6.812781,6.778140,6.794057,6.742300,6.787378
8,18/02/2026,38.403159,6.751257,6.812040,6.798968,6.791234,6.786700,7.020675


In [28]:
cell_pbar = tqdm(total=10, desc='Cell 15/15: Save outputs', leave=False)

# Optional: save outputs
out_dir = Path('production/reassessment')
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(out_dir / 'stepwise_holdout_metrics.csv', index=False)
cell_pbar.update(1)

lgbm_metrics_df.to_csv(out_dir / 'lightgbm_holdout_metrics.csv', index=False)
cell_pbar.update(1)

xgb_metrics_df.to_csv(out_dir / 'xgboost_holdout_metrics.csv', index=False)
cell_pbar.update(1)

cat_metrics_df.to_csv(out_dir / 'catboost_holdout_metrics.csv', index=False)
cell_pbar.update(1)

rf_metrics_df.to_csv(out_dir / 'random_forest_holdout_metrics.csv', index=False)
cell_pbar.update(1)

qr_metrics_df.to_csv(out_dir / 'quantile_regression_holdout_metrics.csv', index=False)
cell_pbar.update(1)

leaderboard_df.to_csv(out_dir / 'model_leaderboard_metrics.csv', index=False)
cell_pbar.update(1)

prediction_comparison_df.to_csv(out_dir / 'dbpredict_all_models_comparison.csv', index=False)
cell_pbar.update(1)

pd.DataFrame({'selected_encoded_feature': selected_features}).to_csv(
    out_dir / 'stepwise_selected_encoded_features.csv', index=False
)
cell_pbar.update(1)

feature_shortlist_df.to_csv(out_dir / 'stepwise_feature_shortlist_for_ml.csv', index=False)
cell_pbar.update(1)
cell_pbar.close()

print(f'Saved outputs to: {out_dir.resolve()}')

Saved outputs to: /Users/arifpras/Library/CloudStorage/Dropbox/perisai/dashboard/model/production/reassessment
